In [ ]:
!pip install -q sentence-transformers faiss-cpu openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 17.2 MB/s eta 0:00:00


                     Implwenting RAG without langChain


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="YOUR_API_KEY",
    default_headers={
        "HTTP-Referer": "https://localhost",   # can be anything
        "X-Title": "RAG-Colab-App"             # your app name
    }
)

In [ ]:
from pypdf import PdfReader

reader=PdfReader("/content/1706.03762v7.pdf")
text=""

for page in reader.pages:
  if page.extract_text():
    text+=page.extract_text()

print(text)

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Ex

In [ ]:
def chunking(text, chunksize=300, overlap=50):
  chunks=[]
  start=0;

  while start<len(text):
    chunks.append(text[start:start+chunksize])
    start=start+chunksize-overlap

  return chunks
document=chunking(text)
print(document)

['Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Par', 'oam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\nlukaszkaiser@google.com\nIlli', 'Kaiser ∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder a', ' best\nperforming models also connect the encoder 

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model= SentenceTransformer('all-MiniLM-L6-V2')
doc_embeddings=embed_model.encode(document)
print(doc_embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[ 0.0441373  -0.01403229  0.01323388 ...  0.00751504  0.01272159
   0.01396974]
 [ 0.01358791 -0.017065   -0.00521904 ...  0.07227447 -0.07449073
  -0.01730011]
 [-0.11821299 -0.12168816  0.07623298 ...  0.1050235   0.03220561
  -0.04716046]
 ...
 [-0.09360445  0.0157775  -0.02804411 ...  0.09456643 -0.02556361
   0.01764213]
 [-0.00984329 -0.00625658  0.01236208 ...  0.11812909 -0.01608389
  -0.00983027]
 [-0.02928909 -0.04962326  0.00845426 ...  0.1273702  -0.00910284
   0.0181121 ]]


In [ ]:
import faiss
import numpy as np

dimension=doc_embeddings.shape[1]

index=faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))


In [ ]:
def ask_rag(query, k=3):
  query_embeddings=embed_model.encode(query)
  query_embeddings = np.array(query_embeddings).reshape(1, -1)
  distances, indices = index.search(query_embeddings, k)
  documents=[document[i] for i in indices[0]]

  context="\n".join(documents)

  prompt=f"""
You are a strict assistant.

Rules:
- Answer ONLY using the context
- If answer is not in context, say "Not found"
- Do NOT guess

Context:
{context}

Question:
{query}
"""

  response=client.chat.completions.create(
      model="google/gemma-4-26b-a4b-it:free",
      messages=[
        {"role": "user", "content": prompt}
      ],
      temperature=0.2
  )
  return response.choices[0].message.content
print(ask_rag("what is a transformer"))


RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-26b-a4b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3CeTOGVW79Dor7QkHSRo822Jdeq'}

In [ ]:
import requests

url = "https://openrouter.ai/api/v1/models"

headers = {
    "Authorization": "Bearer sk-or-v1-xxxxxxxx",
}

response = requests.get(url, headers=headers)
models = response.json()

free_models = [m["id"] for m in models["data"] if ":free" in m["id"]]

print(free_models[:10])

['google/gemma-4-26b-a4b-it:free', 'google/gemma-4-31b-it:free', 'nvidia/nemotron-3-super-120b-a12b:free', 'minimax/minimax-m2.5:free', 'arcee-ai/trinity-large-preview:free', 'liquid/lfm-2.5-1.2b-thinking:free', 'liquid/lfm-2.5-1.2b-instruct:free', 'nvidia/nemotron-3-nano-30b-a3b:free', 'nvidia/nemotron-nano-12b-v2-vl:free', 'qwen/qwen3-next-80b-a3b-instruct:free']


                                            LangChain Integrations

In [3]:
!pip install langchain langchain-community langchain-openai chromadb tiktoken pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 8.0 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/1706.03762v7.pdf")
documents = loader.load()
print(documents)

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle B

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks=splitter.split_documents(documents)
print(len(chunks))

52


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

/tmp/ipykernel_6331/3437838554.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings= HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_community.vectorstores import Chroma

chroma_db= Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="db"
)


In [ ]:
retriver=chroma_db.as_retriever(search_type="similarity", search_kwargs={"k":3})
#

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer ONLY using the provided context.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key="YOUR_API_KEY",
    openai_api_base="https://openrouter.ai/api/v1",
    model="nvidia/nemotron-3-nano-30b-a3b:free"
)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain=(
    {
    "context": retriver | format_docs,
    "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()

    )

In [ ]:
response=rag_chain.invoke("What is a transformer")
print(response)

A transformer is a model architecture thatprocesses sequences using stacks of layers built around multi‑head attention mechanisms. The encoder consists of repeated layers, each containing a multi‑head self‑attention sub‑layer followed by a position‑wise feed‑forward network, with residual connections and layer normalization. The decoder has a similar structure, also using multi‑head self‑attention (allowing each position to attend to all preceding positions) and a second attention sub‑layer that attends over the encoder’s output. Both encoder and decoder stacks are composed of identical layers, produce outputs of dimension \(d_{\text{model}} = 512\), and are illustrated in the overall architecture shown in Figure 1. This description matches the provided context.
